<a href="https://colab.research.google.com/github/justii543/MLpreps/blob/main/CodeVulnerability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [ ]:
!pip install transformers torch datasets pandas numpy scikit-learn

In [ ]:
!git clone https://github.com/DLVulDet/PrimeVul

In [ ]:
import os

# Check folder structure
os.listdir("PrimeVul")

Load CodeBERT

In [ ]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)   # MOVE MODEL TO GPU
model.eval()

Embedding Function

In [ ]:
def get_function_embedding(code_snippet):
    inputs = tokenizer(
        code_snippet,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.squeeze().cpu().numpy()

Load Dataset - with limit

In [ ]:
def load_dataset(file_path, limit=None):
    data = []
    with open(file_path, "r") as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            data.append(json.loads(line))
    return data


train_path = "PrimeVul/primevul_train.jsonl"  # adjust if needed

dataset = load_dataset(train_path, limit=None)  # LIMIT for Colab safety #limiting to 200 was selecting only [1]

print("Loaded samples:", len(dataset))

Load Dataset - Without limit

In [ ]:
def load_dataset(file_path):
    data = []
    with open(file_path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_path = "PrimeVul/primevul_train.jsonl"  # adjust if needed
dataset = load_dataset(train_path)

print("Total samples:", len(dataset))

Collect Balanced Data

In [ ]:
balanced_data = []

count_0 = 0
count_1 = 0
limit_per_class = 100   # you can increase later

for item in dataset:
    label = item["target"]

    if label == 0 and count_0 < limit_per_class:
        balanced_data.append(item)
        count_0 += 1

    elif label == 1 and count_1 < limit_per_class:
        balanced_data.append(item)
        count_1 += 1

    # Stop when both classes collected
    if count_0 >= limit_per_class and count_1 >= limit_per_class:
        break

print("Collected:", len(balanced_data))
print("Class 0:", count_0, "Class 1:", count_1)

Generate Embeddings (M1 Output)

In [ ]:
embeddings = []
labels = []

for item in tqdm(balanced_data):
    code = item["func"]
    label = item["target"]

    emb = get_function_embedding(code)

    embeddings.append(emb)
    labels.append(label)

embeddings = np.array(embeddings)
labels = np.array(labels)

print("Embeddings shape:", embeddings.shape)
print("Label distribution:", dict(zip(*np.unique(labels, return_counts=True))))

Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels   # IMPORTANT
)

Train Model

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear', class_weight='balanced')

svm_model.fit(X_train, y_train)

Evaluate Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = svm_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
code_example = """
int vulnerable(char *input) {
    char buffer[10];
    strcpy(buffer, input);
    return 0;
}
"""

emb = get_function_embedding(code_example)
prediction = svm_model.predict([emb])

print("Prediction:", prediction[0])
# 1 = vulnerable, 0 = safe